# AIS 메시지 타입별 파싱 및 저장

`ais_messages`(원문 테이블)의 모든 행을 `pyais`로 디코딩해서, **메시지 타입별로 별도 테이블**에
전체 필드를 저장한다. VSI는 시/분/초(마이크로초 포함)/RSSI/SNR/UI/Link/Slot으로 분해한다.

- 대상: DB에 존재하는 19종 (`1,3,4,5,6,7,8,9,10,11,12,13,14,15,18,19,20,21` + `24`는 Part A/B로 필드가
  달라 `24a`/`24b` 로 분리)
- `radio` 필드: sync_state는 공통으로 분해. SOTDMA 상세(slot_timeout/sub_message)는 msg_type=1,
  ITDMA 상세(slot_increment/num_slots/keep_flag)는 msg_type=3 에서만 분해한다(원본 notebook 로직 그대로
  확장). 그 외 radio를 가진 타입(4,9,11,18)은 sync_state만 채운다.
- 각 테이블은 `source_id`로 `ais_messages.id`를 참조해 원문을 추적할 수 있다.


## 0. 설정

In [1]:
import time
from collections import defaultdict
import psycopg2
from psycopg2.extras import execute_values
from pyais import decode

DB = dict(host="localhost", port=5432, user="sim_user", password="all4land1!", dbname="ais_analysis_db")

# DDL 등 단발성 작업용(autocommit)
admin_conn = psycopg2.connect(**DB)
admin_conn.autocommit = True

# 서버사이드 커서로 스트리밍 읽기 + 배치 쓰기용(트랜잭션 필요, autocommit=False)
conn = psycopg2.connect(**DB)
conn.autocommit = False
print("DB 접속 OK")

DB 접속 OK


## 1. 스키마 정의
타입별 컬럼 목록(이름, PostgreSQL 타입)을 정의한다.

In [2]:
COMMON_COLS = [
    ("source_id", "BIGINT"),
    ("recv_time", "TIMESTAMP(6)"),
    ("vsi_ui", "SMALLINT"),
    ("vsi_link", "SMALLINT"),
    ("vsi_hour", "SMALLINT"),
    ("vsi_minute", "SMALLINT"),
    ("vsi_second", "NUMERIC(9,6)"),
    ("vsi_slot", "INTEGER"),
    ("vsi_rssi", "SMALLINT"),
    ("vsi_snr", "SMALLINT"),
]

_POS_A = [   # type 1 / 3 공통 베이스 (Position Report Class A)
    ("repeat", "SMALLINT"), ("mmsi", "INTEGER"), ("status", "SMALLINT"), ("turn", "REAL"),
    ("speed", "REAL"), ("accuracy", "BOOLEAN"), ("lon", "DOUBLE PRECISION"), ("lat", "DOUBLE PRECISION"),
    ("course", "REAL"), ("heading", "SMALLINT"), ("second", "SMALLINT"), ("maneuver", "SMALLINT"),
    ("raim", "BOOLEAN"), ("radio", "INTEGER"),
]
_BASE_TIME = [  # type 4 / 11 공통 (Base Station Report / UTC-date response)
    ("repeat", "SMALLINT"), ("mmsi", "INTEGER"), ("year", "SMALLINT"), ("month", "SMALLINT"),
    ("day", "SMALLINT"), ("hour", "SMALLINT"), ("minute", "SMALLINT"), ("second", "SMALLINT"),
    ("accuracy", "BOOLEAN"), ("lon", "DOUBLE PRECISION"), ("lat", "DOUBLE PRECISION"),
    ("epfd", "SMALLINT"), ("raim", "BOOLEAN"), ("radio", "INTEGER"),
]
_ADDR_MSG = [   # type 6 / 12 공통 베이스 (addressed message)
    ("repeat", "SMALLINT"), ("mmsi", "INTEGER"), ("seqno", "SMALLINT"), ("dest_mmsi", "INTEGER"),
    ("retransmit", "BOOLEAN"),
]
_MMSI4 = [  # type 7 / 13 공통 (binary/safety ack)
    ("repeat", "SMALLINT"), ("mmsi", "INTEGER"),
    ("mmsi1", "INTEGER"), ("mmsiseq1", "SMALLINT"), ("mmsi2", "INTEGER"), ("mmsiseq2", "SMALLINT"),
    ("mmsi3", "INTEGER"), ("mmsiseq3", "SMALLINT"), ("mmsi4", "INTEGER"), ("mmsiseq4", "SMALLINT"),
]

TYPE_SCHEMAS = {
    1: _POS_A + [("sync_state", "SMALLINT"), ("slot_timeout", "SMALLINT"), ("sub_message", "INTEGER")],
    3: _POS_A + [("sync_state", "SMALLINT"), ("slot_increment", "INTEGER"), ("num_slots", "SMALLINT"), ("keep_flag", "BOOLEAN")],
    4: _BASE_TIME + [("sync_state", "SMALLINT")],
    5: [
        ("repeat", "SMALLINT"), ("mmsi", "INTEGER"), ("ais_version", "SMALLINT"), ("imo", "INTEGER"),
        ("callsign", "TEXT"), ("shipname", "TEXT"), ("ship_type", "SMALLINT"), ("to_bow", "SMALLINT"),
        ("to_stern", "SMALLINT"), ("to_port", "SMALLINT"), ("to_starboard", "SMALLINT"), ("epfd", "SMALLINT"),
        ("month", "SMALLINT"), ("day", "SMALLINT"), ("hour", "SMALLINT"), ("minute", "SMALLINT"),
        ("draught", "REAL"), ("destination", "TEXT"), ("dte", "BOOLEAN"),
    ],
    6: _ADDR_MSG + [("dac", "INTEGER"), ("fid", "INTEGER"), ("data", "BYTEA")],
    7: _MMSI4,
    8: [("repeat", "SMALLINT"), ("mmsi", "INTEGER"), ("dac", "INTEGER"), ("fid", "INTEGER"), ("data", "BYTEA")],
    9: [
        ("repeat", "SMALLINT"), ("mmsi", "INTEGER"), ("alt", "INTEGER"), ("speed", "REAL"),
        ("accuracy", "BOOLEAN"), ("lon", "DOUBLE PRECISION"), ("lat", "DOUBLE PRECISION"), ("course", "REAL"),
        ("second", "SMALLINT"), ("reserved_1", "INTEGER"), ("dte", "BOOLEAN"), ("assigned", "BOOLEAN"),
        ("raim", "BOOLEAN"), ("radio", "INTEGER"), ("sync_state", "SMALLINT"),
    ],
    10: [("repeat", "SMALLINT"), ("mmsi", "INTEGER"), ("dest_mmsi", "INTEGER")],
    11: _BASE_TIME + [("sync_state", "SMALLINT")],
    12: _ADDR_MSG + [("text", "TEXT")],
    13: _MMSI4,
    14: [("repeat", "SMALLINT"), ("mmsi", "INTEGER"), ("text", "TEXT")],
    15: [
        ("repeat", "SMALLINT"), ("mmsi", "INTEGER"), ("mmsi1", "INTEGER"), ("type1_1", "SMALLINT"),
        ("offset1_1", "INTEGER"), ("type1_2", "SMALLINT"), ("offset1_2", "INTEGER"), ("mmsi2", "INTEGER"),
        ("type2_1", "SMALLINT"), ("offset2_1", "INTEGER"),
    ],
    18: [
        ("repeat", "SMALLINT"), ("mmsi", "INTEGER"), ("reserved_1", "INTEGER"), ("speed", "REAL"),
        ("accuracy", "BOOLEAN"), ("lon", "DOUBLE PRECISION"), ("lat", "DOUBLE PRECISION"), ("course", "REAL"),
        ("heading", "SMALLINT"), ("second", "SMALLINT"), ("reserved_2", "INTEGER"), ("cs", "BOOLEAN"),
        ("display", "BOOLEAN"), ("dsc", "BOOLEAN"), ("band", "BOOLEAN"), ("msg22", "BOOLEAN"),
        ("assigned", "BOOLEAN"), ("raim", "BOOLEAN"), ("radio", "INTEGER"), ("sync_state", "SMALLINT"),
    ],
    19: [
        ("repeat", "SMALLINT"), ("mmsi", "INTEGER"), ("reserved_1", "INTEGER"), ("speed", "REAL"),
        ("accuracy", "BOOLEAN"), ("lon", "DOUBLE PRECISION"), ("lat", "DOUBLE PRECISION"), ("course", "REAL"),
        ("heading", "SMALLINT"), ("second", "SMALLINT"), ("reserved_2", "INTEGER"), ("shipname", "TEXT"),
        ("ship_type", "SMALLINT"), ("to_bow", "SMALLINT"), ("to_stern", "SMALLINT"), ("to_port", "SMALLINT"),
        ("to_starboard", "SMALLINT"), ("epfd", "SMALLINT"), ("raim", "BOOLEAN"), ("dte", "BOOLEAN"),
        ("assigned", "BOOLEAN"),
    ],
    20: [
        ("repeat", "SMALLINT"), ("mmsi", "INTEGER"),
        ("offset1", "INTEGER"), ("number1", "INTEGER"), ("timeout1", "INTEGER"), ("increment1", "INTEGER"),
        ("offset2", "INTEGER"), ("number2", "INTEGER"), ("timeout2", "INTEGER"), ("increment2", "INTEGER"),
        ("offset3", "INTEGER"), ("number3", "INTEGER"), ("timeout3", "INTEGER"), ("increment3", "INTEGER"),
        ("offset4", "INTEGER"), ("number4", "INTEGER"), ("timeout4", "INTEGER"), ("increment4", "INTEGER"),
    ],
    21: [
        ("repeat", "SMALLINT"), ("mmsi", "INTEGER"), ("aid_type", "SMALLINT"), ("name", "TEXT"),
        ("accuracy", "BOOLEAN"), ("lon", "DOUBLE PRECISION"), ("lat", "DOUBLE PRECISION"),
        ("to_bow", "SMALLINT"), ("to_stern", "SMALLINT"), ("to_port", "SMALLINT"), ("to_starboard", "SMALLINT"),
        ("epfd", "SMALLINT"), ("second", "SMALLINT"), ("off_position", "BOOLEAN"), ("reserved_1", "INTEGER"),
        ("raim", "BOOLEAN"), ("virtual_aid", "BOOLEAN"), ("assigned", "BOOLEAN"),
        ("name_ext", "TEXT"), ("full_name", "TEXT"),
    ],
    "24a": [("repeat", "SMALLINT"), ("mmsi", "INTEGER"), ("partno", "SMALLINT"), ("shipname", "TEXT")],
    "24b": [
        ("repeat", "SMALLINT"), ("mmsi", "INTEGER"), ("partno", "SMALLINT"), ("ship_type", "SMALLINT"),
        ("vendorid", "TEXT"), ("model", "SMALLINT"), ("serial", "INTEGER"), ("callsign", "TEXT"),
        ("to_bow", "SMALLINT"), ("to_stern", "SMALLINT"), ("to_port", "SMALLINT"), ("to_starboard", "SMALLINT"),
    ],
}

def table_name(key):
    return f"ais_msg_{key}"

print(f"{len(TYPE_SCHEMAS)}개 테이블 스키마 정의 완료:", [table_name(k) for k in TYPE_SCHEMAS])

20개 테이블 스키마 정의 완료: ['ais_msg_1', 'ais_msg_3', 'ais_msg_4', 'ais_msg_5', 'ais_msg_6', 'ais_msg_7', 'ais_msg_8', 'ais_msg_9', 'ais_msg_10', 'ais_msg_11', 'ais_msg_12', 'ais_msg_13', 'ais_msg_14', 'ais_msg_15', 'ais_msg_18', 'ais_msg_19', 'ais_msg_20', 'ais_msg_21', 'ais_msg_24a', 'ais_msg_24b']


## 2. 테이블 생성 (재실행 안전: DROP 후 재생성)

In [3]:
with admin_conn.cursor() as cur:
    for key, cols in TYPE_SCHEMAS.items():
        tbl = table_name(key)
        col_defs = ",\n    ".join(f"{name} {pgtype}" for name, pgtype in (COMMON_COLS + cols))
        ddl = f"""
        DROP TABLE IF EXISTS {tbl};
        CREATE TABLE {tbl} (
            id BIGSERIAL PRIMARY KEY,
            {col_defs},
            FOREIGN KEY (source_id) REFERENCES ais_messages(id)
        );
        CREATE INDEX idx_{tbl}_recv_time ON {tbl} (recv_time);
        CREATE INDEX idx_{tbl}_mmsi ON {tbl} (mmsi);
        """
        cur.execute(ddl)
        print(f"생성 완료: {tbl} ({len(COMMON_COLS)+len(cols)}개 컬럼)")
print("\n전체 테이블 생성 완료")

생성 완료: ais_msg_1 (27개 컬럼)
생성 완료: ais_msg_3 (28개 컬럼)


생성 완료: ais_msg_4 (25개 컬럼)


생성 완료: ais_msg_5 (29개 컬럼)


생성 완료: ais_msg_6 (18개 컬럼)
생성 완료: ais_msg_7 (20개 컬럼)


생성 완료: ais_msg_8 (15개 컬럼)
생성 완료: ais_msg_9 (25개 컬럼)


...

생성 완료: ais_msg_14 (13개 컬럼)
생성 완료: ais_msg_15 (20개 컬럼)


생성 완료: ais_msg_18 (30개 컬럼)


생성 완료: ais_msg_19 (31개 컬럼)
생성 완료: ais_msg_20 (28개 컬럼)


생성 완료: ais_msg_21 (30개 컬럼)


생성 완료: ais_msg_24a (14개 컬럼)
생성 완료: ais_msg_24b (22개 컬럼)

전체 테이블 생성 완료


## 3. 파싱 함수

In [4]:
import enum

def parse_vsi_fields(vsi):
    try:
        parts = vsi.split(",")
        toa = parts[3]
        return {
            "vsi_ui": int(parts[1]),
            "vsi_link": int(parts[2]),
            "vsi_hour": int(toa[0:2]),
            "vsi_minute": int(toa[2:4]),
            "vsi_second": float(toa[4:]),
            "vsi_slot": int(parts[4]),
            "vsi_rssi": int(parts[5]),
            "vsi_snr": int(parts[6].split("*")[0]),
        }
    except Exception:
        return {k: None for k in
                ["vsi_ui","vsi_link","vsi_hour","vsi_minute","vsi_second","vsi_slot","vsi_rssi","vsi_snr"]}

def parse_comm_state(radio, msg_type):
    r = int(radio)
    sync_state = (r >> 17) & 0x3
    if msg_type == 1:
        return {"sync_state": sync_state, "slot_timeout": (r >> 14) & 0x7, "sub_message": r & 0x3FFF}
    elif msg_type == 3:
        return {"sync_state": sync_state, "slot_increment": (r >> 5) & 0x1FFF,
                "num_slots": (r >> 2) & 0x7, "keep_flag": bool(r & 0x1)}
    else:
        return {"sync_state": sync_state}

def normalize(v):
    """pyais Enum(IntEnum/FloatEnum) 값을 순수 파이썬 primitive로 변환"""
    if isinstance(v, enum.Enum):
        return v.value
    return v

def parse_row(source_id, recv_time, ais_raw, vsi_raw):
    """한 행(ais_messages) -> (table_key, row_dict) 또는 실패 시 (None, 에러메시지)"""
    try:
        parts = [p.encode() for p in ais_raw.split("|")]
        decoded = {k: normalize(v) for k, v in decode(*parts).asdict().items()}
    except Exception as e:
        return None, str(e)

    msg_type = decoded.get("msg_type")
    if msg_type == 24:
        table_key = "24a" if decoded.get("partno") == 0 else "24b"
    else:
        table_key = msg_type

    schema = TYPE_SCHEMAS.get(table_key)
    if schema is None:
        return None, f"미지원 msg_type={msg_type}"

    row = {"source_id": source_id, "recv_time": recv_time}
    row.update(parse_vsi_fields(vsi_raw) if vsi_raw else
               {k: None for k in ["vsi_ui","vsi_link","vsi_hour","vsi_minute","vsi_second","vsi_slot","vsi_rssi","vsi_snr"]})
    for col, _ in schema:
        row[col] = decoded.get(col)
    if msg_type in (1, 3, 4, 9, 11, 18) and "radio" in decoded:
        row.update(parse_comm_state(decoded["radio"], msg_type))

    return table_key, row

print("파싱 함수 정의 완료")

파싱 함수 정의 완료


## 4. 전체 행 파싱 및 타입별 배치 적재

In [5]:
with conn.cursor(name="ais_reader") as read_cur:  # server-side cursor로 스트리밍
    read_cur.itersize = 20000
    read_cur.execute("SELECT id, recv_time, ais_raw, vsi_raw FROM ais_messages ORDER BY id")

    buffers = defaultdict(list)
    col_order = {key: [c for c, _ in (COMMON_COLS + cols)] for key, cols in TYPE_SCHEMAS.items()}
    counts = defaultdict(int)
    failures = []
    t0 = time.time()
    n_processed = 0

    def flush(key):
        if not buffers[key]:
            return
        cols = col_order[key]
        tbl = table_name(key)
        values = [tuple(r.get(c) for c in cols) for r in buffers[key]]
        insert_sql = f"INSERT INTO {tbl} ({', '.join(cols)}) VALUES %s"
        with conn.cursor() as wcur:
            execute_values(wcur, insert_sql, values, page_size=5000)
        counts[key] += len(buffers[key])
        buffers[key].clear()

    for source_id, recv_time, ais_raw, vsi_raw in read_cur:
        table_key, result = parse_row(source_id, recv_time, ais_raw, vsi_raw)
        n_processed += 1
        if table_key is None:
            failures.append((source_id, result))
        else:
            buffers[table_key].append(result)
            if len(buffers[table_key]) >= 5000:
                flush(table_key)
        if n_processed % 100000 == 0:
            print(f"  진행: {n_processed:,} 행 처리 ({time.time()-t0:.1f}s)")

    for key in list(buffers.keys()):
        flush(key)

conn.commit()
print(f"\n총 처리: {n_processed:,} 행  ({time.time()-t0:.1f}s)")
print(f"성공 적재: {sum(counts.values()):,} 행")
print(f"실패: {len(failures)} 행")
if failures:
    print("실패 샘플:", failures[:5])
print("\n타입별 적재 건수:")
for key in sorted(counts, key=str):
    print(f"  {table_name(key):14s} {counts[key]:>8,}")

  진행: 100,000 행 처리 (8.0s)


  진행: 200,000 행 처리 (17.7s)


  진행: 300,000 행 처리 (26.8s)


  진행: 400,000 행 처리 (37.7s)


  진행: 500,000 행 처리 (48.1s)


  진행: 600,000 행 처리 (58.9s)


  진행: 700,000 행 처리 (69.6s)


  진행: 800,000 행 처리 (80.4s)


  진행: 900,000 행 처리 (91.9s)



총 처리: 994,796 행  (119.4s)
성공 적재: 994,796 행
실패: 0 행

타입별 적재 건수:
  ais_msg_1       859,305
  ais_msg_10            2
  ais_msg_11            1
  ais_msg_12           12
  ais_msg_13            2
  ais_msg_14            1
  ais_msg_15        1,253
  ais_msg_18        2,894
  ais_msg_19        1,189
  ais_msg_20        3,324
  ais_msg_21        3,119
  ais_msg_24a         733
  ais_msg_24b         768
  ais_msg_3        77,891
  ais_msg_4         4,448
  ais_msg_5        32,675
  ais_msg_6         2,394
  ais_msg_7         1,169
  ais_msg_8         3,608
  ais_msg_9             8


## 5. 검증

원본 `ais_messages`의 타입별 개수(GROUP BY msg_type)와, 새로 만든 타입별 테이블의 행 수를 대조한다.
(24번은 24a+24b 합으로 비교)

In [6]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM ais_messages WHERE ais_raw IS NOT NULL")
    total_raw = cur.fetchone()[0]

    cur.execute("SELECT msg_type, COUNT(*) FROM ais_messages GROUP BY msg_type ORDER BY msg_type")
    raw_dist = dict(cur.fetchall())

    table_totals = {}
    for key in TYPE_SCHEMAS:
        cur.execute(f"SELECT COUNT(*) FROM {table_name(key)}")
        table_totals[key] = cur.fetchone()[0]

print(f"ais_messages 원문 총계(ais_raw not null): {total_raw:,}")
print(f"파싱 성공 + 실패 = {sum(counts.values()) + len(failures):,}")
print()

mismatch = 0
for mt in sorted(raw_dist):
    if mt == 24:
        parsed = table_totals.get("24a", 0) + table_totals.get("24b", 0)
    else:
        parsed = table_totals.get(mt, 0)
    ok = "OK" if parsed == raw_dist[mt] else "!! 불일치"
    if parsed != raw_dist[mt]:
        mismatch += 1
    print(f"  msg_type={mt:<3} 원문={raw_dist[mt]:>7,}  파싱테이블={parsed:>7,}  {ok}")

print(f"\n불일치 타입 수: {mismatch}")
assert mismatch == 0, "타입별 개수 불일치!"
assert sum(counts.values()) + len(failures) == total_raw, "총 처리 건수 불일치!"
print("\n검증 통과: 모든 메시지가 올바른 타입별 테이블에 누락 없이 저장됨")

ais_messages 원문 총계(ais_raw not null): 994,796
파싱 성공 + 실패 = 994,796

  msg_type=1   원문=859,305  파싱테이블=859,305  OK
  msg_type=3   원문= 77,891  파싱테이블= 77,891  OK
  msg_type=4   원문=  4,448  파싱테이블=  4,448  OK
  msg_type=5   원문= 32,675  파싱테이블= 32,675  OK
  msg_type=6   원문=  2,394  파싱테이블=  2,394  OK
  msg_type=7   원문=  1,169  파싱테이블=  1,169  OK
  msg_type=8   원문=  3,608  파싱테이블=  3,608  OK
  msg_type=9   원문=      8  파싱테이블=      8  OK
  msg_type=10  원문=      2  파싱테이블=      2  OK
  msg_type=11  원문=      1  파싱테이블=      1  OK
  msg_type=12  원문=     12  파싱테이블=     12  OK
  msg_type=13  원문=      2  파싱테이블=      2  OK
  msg_type=14  원문=      1  파싱테이블=      1  OK
  msg_type=15  원문=  1,253  파싱테이블=  1,253  OK
  msg_type=18  원문=  2,894  파싱테이블=  2,894  OK
  msg_type=19  원문=  1,189  파싱테이블=  1,189  OK
  msg_type=20  원문=  3,324  파싱테이블=  3,324  OK
  msg_type=21  원문=  3,119  파싱테이블=  3,119  OK
  msg_type=24  원문=  1,501  파싱테이블=  1,501  OK

불일치 타입 수: 0

검증 통과: 모든 메시지가 올바른 타입별 테이블에 누락 없이 저장됨


### 5-1. 샘플 확인 (type 1, type 5, type 24a/24b)

In [7]:
import pandas as pd
for tbl in ["ais_msg_1", "ais_msg_5", "ais_msg_24a", "ais_msg_24b"]:
    with conn.cursor() as cur:
        cur.execute(f"SELECT * FROM {tbl} ORDER BY id LIMIT 3")
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
    print(f"\n=== {tbl} ===")
    print(pd.DataFrame(rows, columns=cols).to_string(index=False))


=== ais_msg_1 ===
 id  source_id                  recv_time  vsi_ui  vsi_link  vsi_hour  vsi_minute vsi_second  vsi_slot  vsi_rssi  vsi_snr  repeat      mmsi  status   turn  speed  accuracy        lon       lat  course  heading  second  maneuver  raim  radio  sync_state  slot_timeout  sub_message
  1          2 2026-06-09 17:56:42.970000       0         4         8          56  43.576728      1634       -62       52       0 440714100       8    0.0    0.0     False 129.050738 35.119903   154.0       58      42         0 False  34477           0             2         1709
  2          3 2026-06-09 17:56:43.076700       0         5         8          56  43.683395      1638       -74       40       0 440191350       0 -128.0    0.0      True 129.026650 35.086420   158.7      511      42         0  True  34481           0             2         1713
  3          5 2026-06-09 17:56:43.263300       0         7         8          56  43.870021      1645       -86       29       0 440142150  

In [8]:
conn.close()
print("완료")

완료
